# Solving the Krusell-Smith (1998) Model in Julia

This notebook solves the Krusell-Smith (1998) model using value function iteration (VFI). The notebook is structured as follows:
1. Model setup (household, firm, equilibrium approximation).
2. Numerical algorithm description.
3. Julia implementation (VFI + simulation + law-of-motion update).

## 1) Original Model and Equilibrium

### HH problem

Individual state is $s=(k,e,z,\Phi)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $\Phi$ is the distribution of wealth and employment across agents.

Given prices $r(z,\Phi)$ and $w(z,\Phi)$, and law of motion for the distribution $\Phi'$, the Bellman equation is
$$
V(k,e,z,\Phi)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',\Phi')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,\Phi))k + w(z,\Phi)e \\
k'\ge k_{min} \\
\Phi' = \mathcal{H}(\Phi,z,z')
$$

However, the distribution $\Phi$ is infinite-dimensional, so we cannot solve this model numerically. 

Instead, we follow Krusell-Smith and approximate the distribution with a finite set of moments, which here we denote $K$ (aggregate capital). 

## 2) Model with Krusell-Smith Approximation

### HH problem
Individual state is $s=(k,e,z,K)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $K$ is aggregate capital.
Given prices and an aggregate forecasting rule, the Bellman equation is
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
K' = \mathcal{H}(K,z)
$$


### Firms problem
We assume a representative firm with production function:
$$
Y = zK^\alpha N^{1-\alpha},
$$
and competitive factor prices are
$$
r = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w = (1-\alpha)z(K/N)^\alpha.
$$

### Equilibrium definition
A recursive equilibrium is a collection
$$
\{V,\,g,\mathcal{H},\,r,\,w,\,\Phi\},
$$

such that:
1. Given $(r(z,K,L),w(z,K,L))$ and perceived law of motion $\mathcal{H}$, value function $V$ and policy function $g$ solve the household problem.
2. Given $z$ and $r(z,K,L), w(z,K,L)$, firm maximizes profits, in particular:
$$
r(z,K,L) = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w(z,K,L) = (1-\alpha)z(K/N)^\alpha.
$$
3. Given $g$ and shock transitions, we can obtain the actual law of motion for aggregate capital:
    $$
    K' = \mathcal{G}(K,z)
    $$
    The actual law of motion $\mathcal{G}$ is consistent with the perceived law of motion $\mathcal{H}$
4. Market clearing:
$$
K = \int k\,d\Phi(k,e,z,K) \\
N = \int e\,d\Phi(k,e,z,K)
$$

Krusell-Smith approximation specifies a particular functional form for the perceived law of motion $\mathcal{H}$, which is log-linear:
$$
\log K' = a_z + b_z \log K
$$

## 3) General Numerical Algorithm (Inner/Outer Iteration)

We use a nested iteration algorithm.

### Inner loop: solve HH problem
Given coefficients $(a_z,b_z)$, we map $K$ to predicted $K'$ and solve HH problem:
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
\log K' = a_z + b_z \log K
$$

At each state $(k,e,z,K)$ we search over $k'\in\mathcal K$ and compute expected continuation value using transition probabilities for $(z,e)\to(z',e')$.
We iterate until value function converged:
$$
\|V_{new}-V\|_\infty < \varepsilon_V.
$$

### Outer loop: update aggregate law of motion
With the policy function $g(k,e,z,K)$ solved from the inner loop, we:
1. Simulate many households and periods.
2. Construct time series data $\{K_t,z_t\}$.
3. Run state-by-state regressions
$$
\log K_{t+1}=a_z+b_z\log K_t+\epsilon_{t+1},\qquad z\in\{b,g\}.
$$
4. Update perceived coefficients with damping
$$
\theta^{new}=\lambda\theta^{old}+(1-\lambda)\hat\theta,\quad \theta=(a_b,b_b,a_g,b_g).
$$
5. Repeat until stopping criterion is met (in this notebook: both $R^2$ exceed a threshold).


---

# Code Implementation in Julia

## 1) Calibration

### 1) Structural parameters
We use:
- Discount factor: $\beta = 0.99$
- Risk aversion: $\sigma = 1$ (log utility)
- Capital share: $\alpha = 0.36$
- Depreciation: $\delta = 0.025$
- Ad hoc borrowing constraint: $k_{min} = 0$
- Aggregate shock values: $z_b=0.99$, $z_g=1.01$
- Hours of labor supply when employed: $\tilde{l}= 0.3271$

We have these following calibration targets:
- Unemployment rates: $u_b=0.10$, $u_g=0.04$
- Average duration of both good and bad times is eight quarters
- Average duration of unemployment is 1.5 quarters in good times and 2.5 quarters in bad times.

### 2) Aggregate productivity shocks
- Aggregate productivity is two-state:
    $$
    z_t \in \{z_b, z_g\}, \quad z_b=0.99,\; z_g=1.01.
    $$

    The transition matrix is:
    $$
    \Pi_z = \begin{bmatrix} \pi_{bb} & \pi_{bg} \\
    \pi_{gb} & \pi_{gg} \end{bmatrix}
    $$


- Aggregate unemployment rates:
  
    We assume different unemployment rates in the two aggregate states:
    $$
    u_b=0.10,\quad u_g=0.04.
    $$
    In the bad state, 10% of agents are unemployed, while in the good state only 4% are unemployed.

- Effective aggregate labor in each state:
    $$
    L_b=\tilde{l}(1-u_b) = 0.2944, \\
    L_g=\tilde{l}(1-u_g) = 0.3142.
    $$

### 3) Calibrating the aggregate transition matrix.

- First, calibrate the transition matrix for the aggregate shock $z$.

    Average duration of each aggregate state is 8 quarters, we know that:

    The duration of state $s$ can be computed:
    $$
    \text{Duration}_s = \sum_{t=1}^\infty t \cdot (1-\pi_{ss}) \pi_{ss}^{t-1} = \frac{1}{1-\pi_{ss}}
    $$
    Thus
    $$
    \pi_{bb} = \pi_{gg} = 1 - \frac{1}{8} = 0.875, \qquad
    \pi_{bg} = \pi_{gb} = 0.125.
    $$

### 4) Calibrating the joint transition matrix for $(z,e)$.

Then, we calibrate the joint transition matrix for $(z,e)$
  
We assume the individual shock $e$ is correlated with the aggregate shock $z$, thus we need to specify the joint transition matrix for $(z,e)$.
    
Addtionally, we assume that given $z$, transition of individual employment status $e$ is independent cross agents:

Thus, we have $\forall s,s'\in\{b,g\}$:
$$
\pi_{ss'00} +  \pi_{ss'01} = \pi_{ss'10} + \pi_{ss'11} = \pi_{ss'}
$$
and
$$
u_s \frac{\pi_{ss'00}}{\pi_{ss'}} + (1-u_s)\frac{\pi_{ss'10}}{\pi_{ss'}} = u_{s'}
$$

From transition matrix of $z$, we have:
$$
\pi_{bb00} + \pi_{bb01} = 0.875, \\
\pi_{bb10} + \pi_{bb11} = 0.875, \\
\pi_{bg00} + \pi_{bg01} = 0.125, \\
\pi_{bg10} + \pi_{bg11} = 0.125, \\
\pi_{gb00} + \pi_{gb01} = 0.125, \\
\pi_{gb10} + \pi_{gb11} = 0.125, \\
\pi_{gg00} + \pi_{gg01} = 0.875, \\
\pi_{gg10} + \pi_{gg11} = 0.875.
$$

We already know that average unemployment duration is 1.5 quarters in good times and 2.5 quarters in bad times, thus we have:
$$
\pi_{bb00} = 1 - \frac{1}{2.5} = 0.6, \\
\pi_{gg00} = 1 - \frac{1}{1.5} = 1/3.
$$
Implies
$$
\pi_{bb01} = 0.875 - \pi_{bb00} = 0.275, \\
\pi_{gg01} = 0.875 - \pi_{gg00} = 0.542.
$$

Then we impose additional restrictions to solve for the rest of the transition probabilities:
$$
\frac{\pi_{gb00}}{\pi_{bb00}} = 1.25 \frac{\pi_{gb}}{\pi_{bb}} \Rightarrow \\
\pi_{gb00} = 1.25 \cdot \frac{0.125}{0.875} \cdot 0.6 = 0.107, \\
\pi_{gb01} = 0.125 - \pi_{gb00} = 0.018,
$$

Another additional restriction:
$$
\frac{\pi_{bg00}}{\pi_{gg00}} = 0.75 \frac{\pi_{bg}}{\pi_{gg}} \Rightarrow \quad \\
\pi_{bg00} = 0.75 \cdot \frac{0.125}{0.875} \cdot \frac{1}{3} = 0.036, \\
\pi_{bg01} = 0.125 - \pi_{bg00} = 0.089, \\
$$

Remaining probabilities have to be such that unemployment is always constant:
$$
u_s \frac{\pi_{ss'00}}{\pi_{ss'}} + (1-u_s)\frac{\pi_{ss'10}}{\pi_{ss'}} = u_{s'}
$$

Thus, we have:
$$
u_g \frac{\pi_{gg00}}{\pi_{gg}} + (1-u_g)\frac{\pi_{gg10}}{\pi_{gg}} = u_g \\
u_g \frac{\pi_{gb00}}{\pi_{gb}} + (1-u_g)\frac{\pi_{gb10}}{\pi_{gb}} = u_b \\
u_b \frac{\pi_{bb00}}{\pi_{bb}} + (1-u_b)\frac{\pi_{bb10}}{\pi_{bb}} = u_b \\
u_b \frac{\pi_{bg00}}{\pi_{bg}} + (1-u_b)\frac{\pi_{bg10}}{\pi_{bg}} = u_g
$$

Plug in parameters we know and solve for the rest of the transition probabilities:
$$
0.04 \cdot \frac{1/3}{0.875} + 0.96 \cdot \frac{\pi_{gg10}}{0.875} = 0.04 \quad \Rightarrow \quad \pi_{gg10} = 0.022 \quad \Rightarrow \quad \pi_{gg11} = 0.875 - 0.022 = 0.853 \\
0.04 \cdot \frac{0.107}{0.125} + 0.96 \cdot \frac{\pi_{gb10}}{0.125} = 0.10 \quad \Rightarrow \quad \pi_{gb10} = 0.008 \quad \Rightarrow \quad \pi_{gb11} = 0.125 - 0.008 = 0.117 \\
0.10 \cdot \frac{0.6}{0.875} + 0.90 \cdot \frac{\pi_{bb10}}{0.875} = 0.10 \quad \Rightarrow \quad \pi_{bb10} = 0.030 \quad \Rightarrow \quad \pi_{bb11} = 0.875 - 0.030 = 0.845 \\
0.10 \cdot \frac{0.036}{0.125} + 0.90 \cdot \frac{\pi_{bg10}}{0.125} = 0.04 \quad \Rightarrow \quad \pi_{bg10} = 0.002 \quad \Rightarrow \quad \pi_{bg11} = 0.125 - 0.002 = 0.123
$$

Thus, the joint transition matrix for $(z,e)$ is:
$$
P=
\begin{bmatrix}
\pi_{bb00} & \pi_{bb01} & \pi_{bg00} & \pi_{bg01}\\
\pi_{bb10} & \pi_{bb11} & \pi_{bg10} & \pi_{bg11}\\
\pi_{gb00} & \pi_{gb01} & \pi_{gg00} & \pi_{gg01}\\
\pi_{gb10} & \pi_{gb11} & \pi_{gg10} & \pi_{gg11}
\end{bmatrix}.
$$

Substituting in the calibrated values gives

$$
P \approx
\begin{bmatrix}
0.600 & 0.275 & 0.036 & 0.089\\
0.031 & 0.844 & 0.002 & 0.123\\
0.107 & 0.018 & 0.333 & 0.542\\
0.009 & 0.116 & 0.023 & 0.852
\end{bmatrix}.
$$



## 4) Julia Implementation (Appended)

The cells below are appended and keep your earlier markdown sections unchanged.

### Step 1: Packages and Calibrated Parameters

This cell imports packages and defines calibrated parameters.

Perceived aggregate law:
$$\log K_{t+1}=a_z+b_z\log K_t,\quad z\in\{b,g\}.$$

In [8]:
using Random, Statistics, LinearAlgebra, Printf, Plots

In [9]:
Base.@kwdef mutable struct KSParams
    beta::Float64 = 0.99
    sigma::Float64 = 1.0
    alpha::Float64 = 0.36
    delta::Float64 = 0.025
    l_tilde::Float64 = 0.3271

    z_vals::Vector{Float64} = [0.99, 1.01]   # [bad, good]
    u_rate::Vector{Float64} = [0.10, 0.04]
    N_vals::Vector{Float64} = [0.2944, 0.3142]

    Pz::Matrix{Float64} = [0.875 0.125;
                           0.125 0.875]

    # state order: 1=(b,u), 2=(b,e), 3=(g,u), 4=(g,e)
    Pze::Matrix{Float64} = [0.600 0.275 0.036 0.089;
                            0.031 0.844 0.002 0.123;
                            0.107 0.018 0.333 0.542;
                            0.009 0.116 0.023 0.852]

    k_min::Float64 = 0.0
    k_max::Float64 = 40
    nk::Int = 500
    K_min::Float64 = 1.0
    K_max::Float64 = 40
    nK::Int = 60

    N_agents::Int = 5000
    T_sim::Int = 11000
    burn_in::Int = 1000
end

@inline state_index(ie::Int, iz::Int) = (iz - 1) * 2 + ie

util(c, sigma) = c <= 0 ? -1.0e12 : (sigma == 1.0 ? log(c) : (c^(1.0 - sigma) - 1.0) / (1.0 - sigma))

function make_grids(par::KSParams)
    kgrid = collect(range(par.k_min, par.k_max, length = par.nk))
    Kgrid = collect(range(par.K_min, par.K_max, length = par.nK))
    return kgrid, Kgrid
end

function prices(par::KSParams, K::Float64, iz::Int)
    z = par.z_vals[iz]
    N = par.N_vals[iz]
    Kn = K / max(N, 1.0e-12)
    r = par.alpha * z * Kn^(par.alpha - 1.0) - par.delta
    w = (1.0 - par.alpha) * z * Kn^par.alpha
    return r, w
end

function nearest_index(grid::Vector{Float64}, x::Float64)
    j = searchsortedfirst(grid, x)
    if j <= 1
        return 1
    elseif j > length(grid)
        return length(grid)
    else
        return abs(grid[j] - x) < abs(grid[j - 1] - x) ? j : j - 1
    end
end

function linear_interp_weights(grid::Vector{Float64}, x::Float64)
    j = searchsortedfirst(grid, x)
    if j <= 1
        return 1, 1, 0.0
    elseif j > length(grid)
        n = length(grid)
        return n, n, 0.0
    else
        lo = j - 1
        hi = j
        w = (x - grid[lo]) / (grid[hi] - grid[lo])
        return lo, hi, clamp(w, 0.0, 1.0)
    end
end


@inline function cubic_spline_interp_K(V, ikp::Int, iep::Int, izp::Int, iKlo::Int, iKhi::Int, wK::Float64)
    nK = size(V, 2)
    if iKlo == iKhi || iKlo <= 1 || iKhi >= nK
        return (1.0 - wK) * V[ikp, iKlo, iep, izp] + wK * V[ikp, iKhi, iep, izp]
    end

    i0 = iKlo - 1
    i1 = iKlo
    i2 = iKhi
    i3 = iKhi + 1

    y0 = V[ikp, i0, iep, izp]
    y1 = V[ikp, i1, iep, izp]
    y2 = V[ikp, i2, iep, izp]
    y3 = V[ikp, i3, iep, izp]

    t = wK
    a0 = -0.5 * y0 + 1.5 * y1 - 1.5 * y2 + 0.5 * y3
    a1 = y0 - 2.5 * y1 + 2.0 * y2 - 0.5 * y3
    a2 = -0.5 * y0 + 0.5 * y2
    a3 = y1

    v = ((a0 * t + a1) * t + a2) * t + a3

    # Monotone guard: avoid cubic overshoot between the two bracketing nodes
    lo = min(y1, y2)
    hi = max(y1, y2)
    return clamp(v, lo, hi)
end

par = KSParams()
@assert all(abs.(sum(par.Pz, dims = 2) .- 1.0) .< 1e-12)
@assert all(abs.(sum(par.Pze, dims = 2) .- 1.0) .< 1e-12)
println("Step 1 ready.")

Step 1 ready.


### Step 2: Solve HH problem with VFI and Howard PFI

HH problem is:
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
\log K' = a_z + b_z \log K
$$

First, we solve the HH problem with VFI.

In [10]:
function solve_household_vfi(par::KSParams, kgrid, Kgrid, coeff; max_iter = 2000, tol = 1e-6)
    nk, nK = length(kgrid), length(Kgrid)
    V = zeros(nk, nK, 2, 2)
    Vnew = similar(V)
    pol = ones(Int, nk, nK, 2, 2)

    for it in 1:max_iter
        diff = 0.0
        for iz in 1:2
            for iK in 1:nK
                K = Kgrid[iK]
                r, w = prices(par, K, iz)
                Knext = exp(coeff[iz, 1] + coeff[iz, 2] * log(K))
                iKlo, iKhi, wK = linear_interp_weights(Kgrid, Knext)

                for ie in 1:2
                    labor_income = ie == 2 ? w * par.l_tilde : 0.0
                    cur_state = state_index(ie, iz)

                    for ik in 1:nk
                        cash = (1.0 + r) * kgrid[ik] + labor_income
                        best_v = -1.0e18
                        best_ikp = 1

                        for ikp in 1:nk
                            kp = kgrid[ikp]
                            c = cash - kp
                            u = util(c, par.sigma)
                            if u <= -1.0e11
                                continue
                            end

                            ev = 0.0
                            for izp in 1:2
                                for iep in 1:2
                                    nxt_state = state_index(iep, izp)
                                    p = par.Pze[cur_state, nxt_state]
                                    Vcont = cubic_spline_interp_K(V, ikp, iep, izp, iKlo, iKhi, wK)
                                    ev += p * Vcont
                                end
                            end

                            val = u + par.beta * ev
                            if val > best_v
                                best_v = val
                                best_ikp = ikp
                            end
                        end

                        Vnew[ik, iK, ie, iz] = best_v
                        pol[ik, iK, ie, iz] = best_ikp
                        diff = max(diff, abs(best_v - V[ik, iK, ie, iz]))
                    end
                end
            end
        end

        V, Vnew = Vnew, V
        if it % 50 == 0
            @printf("  VFI iter %d, diff = %.3e\n", it, diff)
        end
        if diff < tol
            @printf("  VFI converged at iter %d, diff = %.3e\n", it, diff)
            return V, pol
        end
    end

    @warn "VFI reached max_iter without full convergence"
    return V, pol
end

solve_household_vfi (generic function with 1 method)

Then we solve HH problem with Howard policy function iteration, which should be much faster.

### Algorithm: Howard Policy Function Iteration (PFI)

We solve
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta \mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\}, \quad\text{s.t.}\quad \\
c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
k'\ge k_{min} \\
\log K' = a_z + b_z \log K.
$$

1. **Initialize**
   - Set an initial policy $g_0(k,e,z,K)$.
   - Set an initial value function $V_0(k,e,z,K)$.

2. **Policy Evaluation (fixed policy)**
   - Keep $g_n$ fixed.
   - For each state $(k,e,z,K)$, compute the implied choice $k'=g_n(k,e,z,K)$ and update
   $$
   V_{m+1}(k,e,z,K)=u(c)+\beta\mathbb E\!\left[V_m(k',e',z',K')\mid e,z\right].
   $$
   - Iterate until:
   $$
   \|V_{m+1}-V_m\|_\infty < \varepsilon_V
   $$
   Then set $V^{g_n} = V_m$.


3. **Policy Improvement**
   - After getting the evaluated value function $V^{g_n}$, we can then optimize at each state:
   $$
   g_{n+1}(k,e,z,K) =\arg\max_{k'\in\mathcal K} \left\{ u(c)+\beta\mathbb E\!\left[V^{g_n}(k',e',z',K')\mid e,z\right] \right\} \quad\text{s.t.}\quad \\
   c + k' = (1+r(z,K,L))k + w(z,K,L)e \\
   k'\ge k_{min} \\
   \log K' = a_z + b_z \log K.
   $$
   - Record how many grid points changed in this policy improvement step, keep iterate until:
   $$
   n\_change = 0.
   $$
   n_change counts how many grid points have different policy choices between $g_n$ and $g_{n+1}$.

4. **Convergence Check**
   - If $n\_change = 0$, the policy is stable on the grid (policy convergence).
   - For stricter stopping, also require the evaluation residual to be small.
   - Otherwise set $n\leftarrow n+1$ and repeat Steps 2�?.

Howard PFI is faster than plain VFI because maximization is done only in the improvement step, while evaluation steps are cheap Bellman updates under a fixed policy.


In [11]:
function solve_household_pfi(par::KSParams, kgrid, Kgrid, coeff;
    max_policy_iter = 200, eval_iter = 25, tol = 1e-6, final_eval_iter = 2000)

    nk, nK = length(kgrid), length(Kgrid)
    V = zeros(nk, nK, 2, 2)
    Vnew = similar(V)

    pol = Array{Int}(undef, nk, nK, 2, 2)
    pol_new = similar(pol)

    for iz in 1:2, iK in 1:nK, ie in 1:2, ik in 1:nk
        pol[ik, iK, ie, iz] = ik
    end

    for pit in 1:max_policy_iter
        # Keep previous V to monitor outer-loop value change
        V_prev_pit = copy(V)

        diff_eval = Inf

        # Policy evaluation (fixed policy)
        for pe in 1:eval_iter
            diff_eval = 0.0
            for iz in 1:2
                for iK in 1:nK
                    K = Kgrid[iK]
                    r, w = prices(par, K, iz)
                    Knext = coeff[iz, 1] + coeff[iz, 2] * K
                    iKlo, iKhi, wK = linear_interp_weights(Kgrid, Knext)

                    for ie in 1:2
                        labor_income = ie == 2 ? w * par.l_tilde : 0.0
                        cur_state = state_index(ie, iz)

                        for ik in 1:nk
                            ikp = pol[ik, iK, ie, iz]
                            kp = kgrid[ikp]
                            c = (1.0 + r) * kgrid[ik] + labor_income - kp
                            u = util(c, par.sigma)

                            if u <= -1.0e11
                                Vnew[ik, iK, ie, iz] = -1.0e18
                            else
                                ev = 0.0
                                for izp in 1:2
                                    for iep in 1:2
                                        nxt_state = state_index(iep, izp)
                                        p = par.Pze[cur_state, nxt_state]
                                        Vcont = cubic_spline_interp_K(V, ikp, iep, izp, iKlo, iKhi, wK)
                                        ev += p * Vcont
                                    end
                                end
                                Vnew[ik, iK, ie, iz] = u + par.beta * ev
                            end

                            diff_eval = max(diff_eval, abs(Vnew[ik, iK, ie, iz] - V[ik, iK, ie, iz]))
                        end
                    end
                end
            end

            V, Vnew = Vnew, V
            if diff_eval < tol
                break
            end
        end

        # Policy improvement
        n_change = 0
        for iz in 1:2
            for iK in 1:nK
                K = Kgrid[iK]
                r, w = prices(par, K, iz)
                Knext = coeff[iz, 1] + coeff[iz, 2] * K
                iKlo, iKhi, wK = linear_interp_weights(Kgrid, Knext)

                for ie in 1:2
                    labor_income = ie == 2 ? w * par.l_tilde : 0.0
                    cur_state = state_index(ie, iz)

                    for ik in 1:nk
                        cash = (1.0 + r) * kgrid[ik] + labor_income
                        best_v = -1.0e18
                        best_ikp = pol[ik, iK, ie, iz]

                        for ikp in 1:nk
                            kp = kgrid[ikp]
                            c = cash - kp
                            u = util(c, par.sigma)
                            if u <= -1.0e11
                                continue
                            end

                            ev = 0.0
                            for izp in 1:2
                                for iep in 1:2
                                    nxt_state = state_index(iep, izp)
                                    p = par.Pze[cur_state, nxt_state]
                                    Vcont = cubic_spline_interp_K(V, ikp, iep, izp, iKlo, iKhi, wK)
                                    ev += p * Vcont
                                end
                            end

                            val = u + par.beta * ev
                            if val > best_v
                                best_v = val
                                best_ikp = ikp
                            end
                        end

                        pol_new[ik, iK, ie, iz] = best_ikp
                        n_change += (best_ikp != pol[ik, iK, ie, iz]) ? 1 : 0
                    end
                end
            end
        end

        pol, pol_new = pol_new, pol
        diff_V = maximum(abs.(V .- V_prev_pit))

        if pit % 10 == 0
            @printf("  PFI iter %d, eval diff = %.3e, dV = %.3e, n_change = %d\n",
                    pit, diff_eval, diff_V, n_change)
        end

        # Strict convergence: policy stable + value stable + eval residual small
        if n_change == 0 && diff_eval < tol && diff_V < tol
            @printf("  PFI converged at iter %d, eval diff = %.3e, dV = %.3e\n",
                    pit, diff_eval, diff_V)
            return V, pol
        end

        # If policy is stable but eval residual still large, finish with extra fixed-policy evaluation
        if n_change == 0 && (diff_eval >= tol || diff_V >= tol)
            for fe in 1:final_eval_iter
                diff_eval = 0.0
                for iz in 1:2
                    for iK in 1:nK
                        K = Kgrid[iK]
                        r, w = prices(par, K, iz)
                        Knext = coeff[iz, 1] + coeff[iz, 2] * K
                        iKlo, iKhi, wK = linear_interp_weights(Kgrid, Knext)

                        for ie in 1:2
                            labor_income = ie == 2 ? w * par.l_tilde : 0.0
                            cur_state = state_index(ie, iz)

                            for ik in 1:nk
                                ikp = pol[ik, iK, ie, iz]
                                kp = kgrid[ikp]
                                c = (1.0 + r) * kgrid[ik] + labor_income - kp
                                u = util(c, par.sigma)

                                if u <= -1.0e11
                                    Vnew[ik, iK, ie, iz] = -1.0e18
                                else
                                    ev = 0.0
                                    for izp in 1:2
                                        for iep in 1:2
                                            nxt_state = state_index(iep, izp)
                                            p = par.Pze[cur_state, nxt_state]
                                            Vcont = cubic_spline_interp_K(V, ikp, iep, izp, iKlo, iKhi, wK)
                                            ev += p * Vcont
                                        end
                                    end
                                    Vnew[ik, iK, ie, iz] = u + par.beta * ev
                                end

                                diff_eval = max(diff_eval, abs(Vnew[ik, iK, ie, iz] - V[ik, iK, ie, iz]))
                            end
                        end
                    end
                end

                V, Vnew = Vnew, V
                if diff_eval < tol
                    @printf("  PFI converged after final evaluation: eval diff = %.3e\n", diff_eval)
                    return V, pol
                end
            end

            @warn "PFI policy stabilized but final fixed-policy evaluation did not reach tol."
            return V, pol
        end
    end

    @warn "PFI reached max_policy_iter without full policy convergence"
    return V, pol
end

println("Step 2 ready.")


Step 2 ready.


We compare the result obtained from VFI and Howard PFI, they should be very close. 

And we also compare the computational time. Howard PFI is much faster than VFI, especially when the state space is large.

In [12]:
#=
# 1) grids + coefficients
kgrid, Kgrid = make_grids(par)
coeff = @isdefined(res) ? res.coeff : [0.05 0.95; 0.05 0.95]

# 2) solve HH with both methods (run once each, keep results and time)
t_vfi = @elapsed begin
    V_vfi, pol_vfi = solve_household_vfi(par, kgrid, Kgrid, coeff; max_iter=2000, tol=1e-5)
end

t_pfi = @elapsed begin
    V_pfi, pol_pfi = solve_household_pfi(par, kgrid, Kgrid, coeff; max_policy_iter=2000, eval_iter=25, tol=1e-5)
end

@printf("\nSpeed comparison (seconds): VFI = %.3f, PFI(Howard) = %.3f\n", t_vfi, t_pfi)

# 3) fix one K
K_fixed = 35.0
iK = nearest_index(Kgrid, K_fixed)

# 4) plot k'(k) for each (z,e)
state_titles = Dict(
    (1,1) => "z=bad, e=unemp",
    (1,2) => "z=bad, e=emp",
    (2,1) => "z=good, e=unemp",
    (2,2) => "z=good, e=emp"
)

p = plot(layout=(2,2), size=(1000,700), legend=:topleft)

panel = 1
for iz in 1:2
    for ie in 1:2
        kp_vfi = [kgrid[pol_vfi[ik, iK, ie, iz]] for ik in 1:length(kgrid)]
        kp_pfi = [kgrid[pol_pfi[ik, iK, ie, iz]] for ik in 1:length(kgrid)]

        plot!(p[panel], kgrid, kp_vfi, lw=2, label="VFI")
        plot!(p[panel], kgrid, kp_pfi, lw=2, ls=:dash, label="PFI")
        plot!(p[panel], kgrid, kgrid, lw=1, ls=:dot, color=:black, label="45°")
        xlabel!(p[panel], "k")
        ylabel!(p[panel], "k'")
        title!(p[panel], "$(state_titles[(iz,ie)]), K=$(round(Kgrid[iK], digits=2))")
        panel += 1
    end
end

display(p)
=#

### Step 3: Simulate and Update Perceived Law (Interpolation Version)

Simulate economy, estimate by state:
$$\log K_{t+1}=a_z+b_z\log K_t+\varepsilon_{t+1},$$
update with damping, and iterate.

Convergence criterion:
$$R^2_{bad}>0.9999\quad\text{and}\quad R^2_{good}>0.9999.$$

In [13]:
function draw_next_z(iz::Int, Pz::Matrix{Float64}, rng)
    udraw = rand(rng)
    return udraw <= Pz[iz, 1] ? 1 : 2
end

function p_emp_next(par::KSParams, ie::Int, iz::Int, izp::Int)
    cur_state = state_index(ie, iz)
    pz = par.Pz[iz, izp]
    if pz <= 1.0e-12
        return ie == 2 ? 1.0 : 0.0
    end
    emp_state = state_index(2, izp)
    p_emp = par.Pze[cur_state, emp_state] / pz
    return clamp(p_emp, 0.0, 1.0)
end

function simulate_economy(par::KSParams, kgrid, Kgrid, pol, coeff; seed = 1234)
    rng = MersenneTwister(seed)
    T, N = par.T_sim, par.N_agents

    k_idx = fill(cld(length(kgrid), 2), N)
    e_state = [rand(rng) < (1.0 - par.u_rate[2]) ? 2 : 1 for _ in 1:N]
    z_path = ones(Int, T + 1)
    z_path[1] = 2

    K_series = zeros(T + 1)
    K_series[1] = mean(kgrid[k_idx])

    for t in 1:T
        iz = z_path[t]
        Kt = mean(kgrid[k_idx])
        K_series[t] = Kt
        iKlo, iKhi, wK = linear_interp_weights(Kgrid, Kt)

        izp = draw_next_z(iz, par.Pz, rng)
        z_path[t + 1] = izp

        k_next_idx = similar(k_idx)
        for i in 1:N
            ie = e_state[i]
            ik = k_idx[i]

            kp_lo = kgrid[pol[ik, iKlo, ie, iz]]
            kp_hi = kgrid[pol[ik, iKhi, ie, iz]]
            kp_star = (1.0 - wK) * kp_lo + wK * kp_hi
            ik_lo, ik_hi, wk = linear_interp_weights(kgrid, kp_star)
            ikp = rand(rng) < wk ? ik_hi : ik_lo
            k_next_idx[i] = ikp

            p_emp = p_emp_next(par, ie, iz, izp)
            e_state[i] = rand(rng) < p_emp ? 2 : 1
        end

        k_idx = k_next_idx
        K_series[t + 1] = mean(kgrid[k_idx])
    end

    return K_series, z_path
end


function fit_law_of_motion(K_series, z_path; burn_in = 1000)
    coeff = zeros(2, 2)
    r2 = zeros(2)

    for z in 1:2
        idx = [t for t in burn_in:(length(K_series) - 1) if z_path[t] == z]
        x = K_series[idx]
        y = K_series[idx .+ 1]

        X = hcat(ones(length(x)), x)
        b = X \ y
        coeff[z, :] .= b

        yhat = X * b
        ssr = sum((y .- yhat) .^ 2)
        sst = sum((y .- mean(y)) .^ 2)
        r2[z] = 1.0 - ssr / max(sst, 1.0e-12)
    end

    return coeff, r2
end


function solve_ks(par::KSParams; coeff_init = [0.05 0.95; 0.05 0.95], max_outer = 30, damping = 0.9, r2_tol = 0.9999, base_seed = 1234, vary_seed = true)
    kgrid, Kgrid = make_grids(par)
    coeff = copy(coeff_init)

    V = zeros(length(kgrid), length(Kgrid), 2, 2)
    pol = ones(Int, length(kgrid), length(Kgrid), 2, 2)
    K_series = zeros(par.T_sim + 1)
    z_path = ones(Int, par.T_sim + 1)
    r2 = zeros(2)

    for out in 1:max_outer
        @printf("\n=== Outer iteration %d ===\n", out)

        V, pol = solve_household_pfi(par, kgrid, Kgrid, coeff, tol = 1e-6, max_policy_iter = 2000, eval_iter = 25)
        seed_used = vary_seed ? (base_seed + out) : base_seed
        K_series, z_path = simulate_economy(par, kgrid, Kgrid, pol, coeff; seed = seed_used)
        new_coeff, r2 = fit_law_of_motion(K_series, z_path; burn_in = par.burn_in)

        gap = maximum(abs.(new_coeff .- coeff))

        @printf("  bad state : a = %.4f, b = %.4f, R2 = %.6f\n", new_coeff[1,1], new_coeff[1,2], r2[1])
        @printf("  good state: a = %.4f, b = %.4f, R2 = %.6f\n", new_coeff[2,1], new_coeff[2,2], r2[2])
        @printf("  max coefficient gap = %.4e\n", gap)

        coeff .= damping .* coeff .+ (1.0 - damping) .* new_coeff

        if (r2[1] > r2_tol) && (r2[2] > r2_tol)
            @printf("Converged: outer loop stopped at iter %d (R2 criterion met).\n", out)
            return (; coeff, r2, V, pol, kgrid, Kgrid, K_series, z_path)
        end
    end

    @warn "Outer loop reached max_outer without meeting R2 threshold."
    return (; coeff, r2, V, pol, kgrid, Kgrid, K_series, z_path)
end


println("Step 3 ready.")

Step 3 ready.


### Step 4: Run the Full KS Solver

Run and report
$$\log K' = a_b + b_b\log K\; (z=b),\qquad \log K' = a_g + b_g\log K\; (z=g).$$

In [ ]:
par = KSParams()
res = solve_ks(par; coeff_init = [0.05 0.95; 0.05 0.95], max_outer = 20, damping = 0.9, r2_tol = 0.9999, vary_seed = false)

println("\nFinal perceived law of motion (rows: bad, good):")
println("[a_z  b_z] =")
display(res.coeff)
println("R2 by state = ", res.r2)

println("\nBad-state law:  log(K') = $(round(res.coeff[1,1], digits=4)) + $(round(res.coeff[1,2], digits=4))*log(K)")
println("Good-state law: log(K') = $(round(res.coeff[2,1], digits=4)) + $(round(res.coeff[2,2], digits=4))*log(K)")


=== Outer iteration 1 ===
  PFI iter 10, eval diff = 8.000e+00, dV = 5.727e+15, n_change = 17780
  PFI converged after final evaluation: eval diff = 9.537e-07
  bad state : a = 0.6019, b = 0.9683, R2 = 0.999881
  good state: a = 0.8973, b = 0.9580, R2 = 0.999923
  max coefficient gap = 8.4732e-01

=== Outer iteration 2 ===
  PFI iter 10, eval diff = 4.000e+00, dV = 5.725e+15, n_change = 22682
  PFI converged after final evaluation: eval diff = 6.407e-07
  bad state : a = 0.6070, b = 0.9691, R2 = 0.999887
  good state: a = 0.8785, b = 0.9603, R2 = 0.999931
  max coefficient gap = 7.4375e-01

=== Outer iteration 3 ===
  PFI iter 10, eval diff = 2.000e+00, dV = 1.049e+16, n_change = 33899
  PFI converged after final evaluation: eval diff = 9.537e-07
  bad state : a = 0.6242, b = 0.9686, R2 = 0.999887
  good state: a = 0.8548, b = 0.9619, R2 = 0.999931
  max coefficient gap = 6.4567e-01

=== Outer iteration 4 ===
  PFI iter 10, eval diff = 1.600e+01, dV = 4.780e+15, n_change = 22604
  PFI

Previously, when I tried to run more outer iterations, the $R^2$ for both states are very close to 0.9999, but not greater than 0.9999.

Then I realized this should come from the numerical error in interpolation and grid discretization.

After I increase the number of grid points and use more accurate interpolation method, I can get $R^2$ closer to 0.9999 for both states.